# Profilage des Athlètes Cyclistes (Cohorte de 7 athlètes – Données brutes 1 Hz)

Pipeline complet d'extraction, de nettoyage, de modélisation physiologique et de profilage :

1. **Signature 1 — Moteur bioénergétique (Dualité Brut & Relatif — Valenzuela et al., 2022)** :
   - *Brut* : classification par Z-scores et probabilités Softmax sur $(\text{Poids}_{\text{kg}},\; CP_{\text{W}},\; W'_{\text{kJ}})$ vis-à-vis des 5 profils de référence Valenzuela.
   - *Relatif* : classification par Z-scores et probabilités Softmax sur $(\text{Poids}_{\text{kg}},\; CP_{\text{W/kg}},\; W'_{\text{J/kg}})$ dérivé par méthode delta (incluant le poids en kg).
2. **Signature 2 — Profil de puissance courte durée (5 s, 30 s, 60 s & CP)** :
   - Visualisation directe des 7 courbes sur 4 points clés (`5 s`, `30 s`, `60 s`, `CP`) avec deux graphiques côte à côte : **valeurs brutes (W)** et **valeurs relatives au poids (W/kg)**.
3. **Signature 3 — Performance en montée (`slope > 4 %`) vs plat (`-1 % <= slope <= 1 %`)** :
   - Puissance moyenne pondérée en montée (`W` et `W/kg`) et ratio `Montée / Plat`.
4. **Dérive cardiaque (Découplage aérobie $Pw:Hr$)** :
   - Évolution de la dérive cardiaque ($\Delta (W / \text{FC})$ entre la 1re et la 2e moitié des sorties $\ge 45\text{ min}$) pour évaluer l'endurance aérobie.
5. **Classification Ascendante Hiérarchique (CAH – Ward)** sur l'ensemble des indicateurs clés de profilage.

In [ ]:
import json
import math
import os
import re
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.optimize import curve_fit
from scipy.spatial.distance import pdist, squareform

# ---------- Paramètres généraux ----------
DOSSIER_SORTIES = Path("Data_Cyclist")
OUTPUT_DIR = Path("output_profiles_json")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FC_MAX = {}               # ex : {"cyclist1": 192}. Sinon : max observé sur ses sorties
HR_MIN_VALIDE = 30
HR_MAX_VALIDE = 230
PAUSE_MAX_S = 10

# Seuils pente (%)
SLOPE_MIN_VALIDE = -25.0
SLOPE_MAX_VALIDE = 30.0
SLOPE_FLAT_ABS = 1.0      # Plat : -1% <= slope <= 1%
SLOPE_CLIMB_MIN = 4.0     # Montée : slope > 4%
MIN_SEGMENT_S = 60.0      # Temps effectif minimal (s) par sortie pour valider un segment

# Seuil dérive cardiaque
MIN_DRIFT_DURATION_S = 2700.0  # 45 min minimum de roulage actif pour calculer la dérive cardiaque

KEEP_COLS = ["secs", "km", "hr", "watts", "slope"]

MMP_DURATIONS = [5, 30, 60, 180, 300, 480, 1200, 3600]
RECENT_WINDOW_DAYS = 90
N_WORKERS = min(8, os.cpu_count() or 4)

# =============================================================================
# 1. RÉFÉRENTIEL LITTÉRATURE (Valenzuela et al., 2022)
# =============================================================================
VALENZUELA_5_ARCHETYPES = {
    "Grimpeur": {
        "english_label": "Climber",
        "poids_kg": (63.2, 4.4),
        "cp_watts": (354.1, 24.6),
        "w_prime_kj": (23.0, 1.3),
    },
    "Rouleur_CLM": {
        "english_label": "Time trialist",
        "poids_kg": (72.6, 5.4),
        "cp_watts": (395.3, 31.8),
        "w_prime_kj": (22.0, 2.7),
    },
    "Polyvalent_Puncheur": {
        "english_label": "All-rounder",
        "poids_kg": (69.5, 5.5),
        "cp_watts": (361.1, 31.7),
        "w_prime_kj": (30.1, 0.2),
    },
    "Sprinteur": {
        "english_label": "Sprinter",
        "poids_kg": (80.2, 6.5),
        "cp_watts": (381.3, 23.7),
        "w_prime_kj": (38.6, 4.5),
    },
    "Classement_General": {
        "english_label": "GC-contender",
        "poids_kg": (63.8, 4.5),
        "cp_watts": (388.6, 24.0),
        "w_prime_kj": (18.1, 2.3),
    },
}


def build_valenzuela_relative_archetypes(base_archetypes: dict, rho: float = 0.5) -> dict:
    """Dérive les paramètres relatifs (CP en W/kg, W' en J/kg) par méthode delta."""
    rel_archetypes = {}
    for cat_name, params in base_archetypes.items():
        mu_w, sd_w = params["poids_kg"]
        mu_cp, sd_cp = params["cp_watts"]
        mu_wp, sd_wp = params["w_prime_kj"]

        mu_cp_wkg = mu_cp / mu_w
        cv_w = sd_w / mu_w
        cv_cp = sd_cp / mu_cp
        sd_cp_wkg = mu_cp_wkg * np.sqrt(max(1e-6, cv_cp**2 + cv_w**2 - 2.0 * rho * cv_cp * cv_w))

        mu_wp_jkg = (mu_wp * 1000.0) / mu_w
        cv_wp = sd_wp / mu_wp
        sd_wp_jkg = mu_wp_jkg * np.sqrt(max(1e-6, cv_wp**2 + cv_w**2 - 2.0 * rho * cv_wp * cv_w))

        rel_archetypes[cat_name] = {
            **params,
            "cp_w_kg": (round(float(mu_cp_wkg), 4), round(float(sd_cp_wkg), 4)),
            "w_prime_j_kg": (round(float(mu_wp_jkg), 2), round(float(sd_wp_jkg), 2)),
        }
    return rel_archetypes


VALENZUELA_5_ARCHETYPES_REL = build_valenzuela_relative_archetypes(VALENZUELA_5_ARCHETYPES)

# Références P90 peloton pro (Valenzuela et al., 2022)
CAH_REFERENCE_BENCHMARKS = {
    "mmp_5s_w_kg": 20.83,
    "mmp_30s_w_kg": 14.15,
    "mmp_60s_w_kg": 11.33,
    "mmp_300s_w_kg": 7.65,
    "mmp_1200s_w_kg": 6.59,
    "mmp_3600s_w_kg": 5.76,
}

# Profils choisis manuellement par athlète (exportés dans les JSON et affichés sur le dendrogramme)
PROFILS_CHOISIS = {
    "cyclist1": "rouleur",
    "cyclist2": "puncheur",
    "cyclist3": "puncheur",
    "cyclist4": "puncheur polyvalent",
    "cyclist5": "polyvalent",
    "cyclist6": "rouleur",
    "cyclist7": "rouleur",
}

## 1. Lecture `float32` et nettoyage des sorties 1 Hz (incluant `slope` et `hr`)

In [ ]:
def date_depuis_nom(stem: str) -> pd.Timestamp:
    """Extrait la date de la sortie depuis le nom de fichier (YYYY_MM_DD ou YYYY-MM-DD)."""
    m = re.match(r"(\d{4})_(\d{2})_(\d{2})", stem)
    if m:
        return pd.Timestamp(int(m[1]), int(m[2]), int(m[3]))
    m_iso = re.search(r"(20\d{2})[-_](0[1-9]|1[0-2])[-_](0[1-9]|[12]\d|3[01])", stem)
    return pd.Timestamp(int(m_iso[1]), int(m_iso[2]), int(m_iso[3])) if m_iso else pd.NaT


def lire_fc_repos(dossier: Path) -> float:
    """Lit la FC de repos si un fichier *fc_repos*.csv existe dans le dossier du cycliste."""
    fichiers_repos = sorted(dossier.rglob("*fc_repos*.csv"))
    if not fichiers_repos:
        return np.nan
    try:
        df_r = pd.read_csv(fichiers_repos[0])
        for col in df_r.columns:
            if any(k in col.lower() for k in ["fc", "hr", "repos", "rest"]):
                vals = pd.to_numeric(df_r[col], errors="coerce").dropna()
                vals = vals[(vals >= HR_MIN_VALIDE) & (vals <= 120)]
                if not vals.empty:
                    return float(vals.median())
    except Exception:
        pass
    return np.nan


def lire_et_nettoyer_sortie(f: Path) -> pd.DataFrame:
    """Charge un export 1 Hz en float32, plafonne les pauses à PAUSE_MAX_S et nettoie les capteurs."""
    df = pd.read_csv(f, usecols=lambda c: c in KEEP_COLS, dtype="float32", engine="c")
    if df.empty or "secs" not in df.columns or "watts" not in df.columns:
        return pd.DataFrame()

    df = df.dropna(subset=["secs"]).sort_values("secs").reset_index(drop=True)
    if len(df) < 2:
        return pd.DataFrame()

    df["dt"] = df["secs"].diff().fillna(0).clip(lower=0, upper=PAUSE_MAX_S)
    df["watts"] = df["watts"].clip(lower=0).fillna(0)
    if "hr" in df.columns:
        df.loc[(df["hr"] < HR_MIN_VALIDE) | (df["hr"] > HR_MAX_VALIDE), "hr"] = np.nan
    if "slope" in df.columns:
        df.loc[(df["slope"] < SLOPE_MIN_VALIDE) | (df["slope"] > SLOPE_MAX_VALIDE), "slope"] = np.nan

    return df


## 2. Extraction des MMP, des métriques montée/plat et de la dérive cardiaque par sortie (Passe 1)

In [ ]:
def _weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    valid = values.notna() & weights.notna() & (weights > 0)
    w_sum = float(weights[valid].sum())
    if w_sum <= 0:
        return np.nan
    return float((values[valid] * weights[valid]).sum() / w_sum)


def compute_cardiac_drift_pct(df: pd.DataFrame) -> float:
    """
    Calcule la dérive cardiaque (découplage aérobie Pw:Hr en %) sur les séances >= 45 min :
    ((EF_1ere_moitie - EF_2eme_moitie) / EF_1ere_moitie) * 100 avec EF = Watts / FC.
    """
    if "hr" not in df.columns or "watts" not in df.columns:
        return np.nan

    active = df[(df["dt"] > 0) & (df["watts"] > 30) & df["hr"].notna() & (df["hr"] > 60)].copy()
    total_active_s = float(active["dt"].sum())
    if total_active_s < MIN_DRIFT_DURATION_S:
        return np.nan

    cum_t = active["dt"].cumsum()
    half_t = total_active_s / 2.0

    first_half = active[cum_t <= half_t]
    second_half = active[cum_t > half_t]

    if first_half.empty or second_half.empty:
        return np.nan

    w1 = _weighted_mean(first_half["watts"], first_half["dt"])
    hr1 = _weighted_mean(first_half["hr"], first_half["dt"])
    w2 = _weighted_mean(second_half["watts"], second_half["dt"])
    hr2 = _weighted_mean(second_half["hr"], second_half["dt"])

    if any(pd.isna(x) or x <= 0 for x in [w1, hr1, w2, hr2]):
        return np.nan

    ef1 = w1 / hr1
    ef2 = w2 / hr2
    drift_pct = ((ef1 - ef2) / ef1) * 100.0

    if drift_pct < -25.0 or drift_pct > 45.0:
        return np.nan
    return float(drift_pct)


def lire_sortie_profil(f: Path) -> dict:
    """Calcule les MMP, la puissance moyenne, la dérive cardiaque et les métriques de montée/plat."""
    df = lire_et_nettoyer_sortie(f)
    if df.empty:
        return {}

    moving_s = float(df["dt"].sum())
    if moving_s < 120:
        return {}

    stem = f.stem
    date = date_depuis_nom(stem)
    if pd.isna(date):
        date = pd.Timestamp(datetime.fromtimestamp(f.stat().st_mtime).date())

    sec_min, sec_max = int(df["secs"].min()), int(df["secs"].max())
    if (sec_max - sec_min + 1) == len(df) and (df["secs"].diff().dropna() == 1).all():
        watts_1hz = df["watts"].astype(np.float64)
    else:
        grid = pd.Series(0.0, index=np.arange(sec_min, sec_max + 1, dtype=np.int64))
        dedup = df.groupby(df["secs"].round().astype(np.int64))["watts"].mean()
        grid.loc[dedup.index] = dedup.to_numpy(dtype=np.float64)
        watts_1hz = grid

    mmp = {
        f"mmp_{d}s": (
            float(watts_1hz.rolling(window=d, min_periods=d).mean().max())
            if len(watts_1hz) >= d else np.nan
        )
        for d in MMP_DURATIONS
    }

    avg_watts = _weighted_mean(df["watts"], df["dt"])
    avg_hr = _weighted_mean(df["hr"], df["dt"]) if "hr" in df.columns else np.nan
    cardiac_drift_pct = compute_cardiac_drift_pct(df)

    # Métriques terrain : Montée (slope > 4%) vs Plat (-1% <= slope <= 1%)
    watts_climb = np.nan
    watts_flat = np.nan
    climb_flat_ratio = np.nan
    climb_time_min = 0.0

    if "slope" in df.columns:
        mask_climb = df["slope"] > SLOPE_CLIMB_MIN
        mask_flat = df["slope"].between(-SLOPE_FLAT_ABS, SLOPE_FLAT_ABS)

        dt_climb = float(df.loc[mask_climb, "dt"].sum())
        dt_flat = float(df.loc[mask_flat, "dt"].sum())
        climb_time_min = dt_climb / 60.0

        if dt_climb >= MIN_SEGMENT_S:
            watts_climb = _weighted_mean(df.loc[mask_climb, "watts"], df.loc[mask_climb, "dt"])

        if dt_flat >= MIN_SEGMENT_S:
            watts_flat = _weighted_mean(df.loc[mask_flat, "watts"], df.loc[mask_flat, "dt"])

        if pd.notna(watts_climb) and pd.notna(watts_flat) and watts_flat > 20.0:
            climb_flat_ratio = float(watts_climb / watts_flat)

    return {
        "seance": stem,
        "date": date,
        "annee": int(date.year),
        "moving_min": float(moving_s / 60.0),
        "distance_km": float(df["km"].max() - df["km"].min()) if "km" in df else np.nan,
        "work_kj": float((df["watts"] * df["dt"]).sum() / 1000.0),
        "avg_watts": avg_watts,
        "avg_hr": avg_hr,
        "hr_max": float(df["hr"].max()) if "hr" in df else np.nan,
        "cardiac_drift_pct": cardiac_drift_pct,
        "climb_time_min": climb_time_min,
        "watts_climb": watts_climb,
        "watts_flat": watts_flat,
        "climb_flat_ratio": climb_flat_ratio,
        **mmp,
    }


## 3. Modélisation CP / $W'$, Signatures 1-2-3 et Dérive Cardiaque

In [ ]:
def fit_critical_power(mmp_dict: dict) -> tuple[float, float]:
    """Ajuste le modèle hyperbolique P(t) = CP + W'/t sur les efforts de 3 min à 20 min."""
    fit_durations = np.array([180, 300, 480, 1200], dtype=float)
    powers = np.array([mmp_dict.get(f"mmp_{int(d)}s", np.nan) for d in fit_durations], dtype=float)
    valid = ~np.isnan(powers)
    if valid.sum() < 2:
        return np.nan, np.nan

    model = lambda t, cp, w_prime: cp + w_prime / t
    try:
        popt, _ = curve_fit(
            model,
            fit_durations[valid],
            powers[valid],
            p0=[250.0, 15000.0],
            bounds=([50.0, 1000.0], [650.0, 60000.0]),
        )
        return float(popt[0]), float(popt[1])
    except Exception:
        return np.nan, np.nan


def _zscore_softmax_classification(
    athlete_vec: dict,
    archetypes_ref: dict,
    min_cv: float = 0.05,
) -> dict:
    """Classification par distance RMS Z-score et Softmax gaussien vis-à-vis d'un référentiel."""
    if not athlete_vec:
        return {
            "dominant_profile": "Indetermine",
            "min_rms_z_distance": np.nan,
            "dominant_probability_pct": np.nan,
            "second_profile": None,
            "second_probability_pct": np.nan,
            "variables_used": [],
            "rms_z_distances": {},
            "membership_probabilities_pct": {},
            "z_scores_detail": {},
            "hybrid_profile_note": "Données insuffisantes",
        }

    z_scores_detail = {}
    rms_z_distances = {}
    sq_sums = {}

    for cat_name, params in archetypes_ref.items():
        cat_z = {}
        sq = 0.0
        for feat, val in athlete_vec.items():
            mu, sd = params[feat]
            sd_reg = max(float(sd), float(min_cv * mu))
            z = float((val - mu) / sd_reg)
            cat_z[feat] = z
            sq += z ** 2

        z_scores_detail[cat_name] = cat_z
        sq_sums[cat_name] = float(sq)
        rms_z_distances[cat_name] = float(np.sqrt(sq / len(athlete_vec)))

    min_sq = min(sq_sums.values())
    unnorm_probs = {k: float(np.exp(-0.5 * (s - min_sq))) for k, s in sq_sums.items()}
    denom = sum(unnorm_probs.values())
    membership_probs_pct = {
        k: float((v / denom) * 100.0) if denom > 0 else np.nan
        for k, v in unnorm_probs.items()
    }

    ranked = sorted(rms_z_distances.keys(), key=lambda k: rms_z_distances[k])
    dominant = ranked[0]
    second = ranked[1] if len(ranked) > 1 else None

    if second and membership_probs_pct[second] >= 15.0:
        hybrid_note = (
            f"{dominant} ({membership_probs_pct[dominant]:.1f}%) / "
            f"{second} ({membership_probs_pct[second]:.1f}%)"
        )
    else:
        hybrid_note = f"{dominant} pur ({membership_probs_pct[dominant]:.1f}%)"

    return {
        "dominant_profile": dominant,
        "min_rms_z_distance": rms_z_distances[dominant],
        "dominant_probability_pct": membership_probs_pct[dominant],
        "second_profile": second,
        "second_probability_pct": membership_probs_pct[second] if second else np.nan,
        "variables_used": list(athlete_vec.keys()),
        "rms_z_distances": rms_z_distances,
        "membership_probabilities_pct": membership_probs_pct,
        "z_scores_detail": z_scores_detail,
        "hybrid_profile_note": hybrid_note,
    }


def classify_bioenergetic_valenzuela_dual(
    weight_kg: float,
    cp_watts: float,
    w_prime_joules: float,
    min_cv: float = 0.05,
) -> dict:
    """Signature 1 — Moteur bioénergétique (Dualité Brut & Relatif — Valenzuela et al., 2022)."""
    raw_vec = {}
    if pd.notna(weight_kg) and weight_kg > 0:
        raw_vec["poids_kg"] = float(weight_kg)
    if pd.notna(cp_watts) and cp_watts > 0:
        raw_vec["cp_watts"] = float(cp_watts)
    if pd.notna(w_prime_joules) and w_prime_joules > 0:
        raw_vec["w_prime_kj"] = float(w_prime_joules / 1000.0)
    raw_res = _zscore_softmax_classification(raw_vec, VALENZUELA_5_ARCHETYPES, min_cv=min_cv)

    rel_vec = {}
    if pd.notna(weight_kg) and weight_kg > 0:
        rel_vec["poids_kg"] = float(weight_kg)
        if pd.notna(cp_watts) and cp_watts > 0:
            rel_vec["cp_w_kg"] = float(cp_watts / weight_kg)
        if pd.notna(w_prime_joules) and w_prime_joules > 0:
            rel_vec["w_prime_j_kg"] = float(w_prime_joules / weight_kg)
    rel_res = _zscore_softmax_classification(rel_vec, VALENZUELA_5_ARCHETYPES_REL, min_cv=min_cv)

    return {
        "raw": raw_res,
        "relative": rel_res,
    }


def extract_sig2_4points(
    best_mmp: dict,
    relative_metrics: dict,
    cp_watts: float,
) -> dict:
    """Signature 2 — Les 4 points clés de puissance (5 s, 30 s, 60 s, CP) en brut (W) et relatif (W/kg)."""
    return {
        "raw_watts": {
            "5s": float(best_mmp.get("mmp_5s", np.nan)),
            "30s": float(best_mmp.get("mmp_30s", np.nan)),
            "60s": float(best_mmp.get("mmp_60s", np.nan)),
            "CP": float(cp_watts) if pd.notna(cp_watts) else np.nan,
        },
        "relative_w_kg": {
            "5s": float(relative_metrics.get("mmp_5s_w_kg", np.nan)),
            "30s": float(relative_metrics.get("mmp_30s_w_kg", np.nan)),
            "60s": float(relative_metrics.get("mmp_60s_w_kg", np.nan)),
            "CP": float(relative_metrics.get("cp_w_kg", np.nan)),
        },
    }


def extract_climbing_metrics(rides_df: pd.DataFrame, weight_kg: float) -> dict:
    """Indicateurs de performance en montée (slope > 4%) et sur le plat (-1% <= slope <= 1%)."""
    valid_climb = rides_df["watts_climb"].dropna()
    valid_ratio = rides_df["climb_flat_ratio"].dropna()

    med_watts_climb = float(valid_climb.median()) if not valid_climb.empty else np.nan
    max_watts_climb = float(valid_climb.max()) if not valid_climb.empty else np.nan
    med_watts_flat = float(rides_df["watts_flat"].dropna().median()) if not rides_df["watts_flat"].dropna().empty else np.nan
    med_ratio_cf = float(valid_ratio.median()) if not valid_ratio.empty else np.nan

    med_w_kg_climb = (
        float(med_watts_climb / weight_kg)
        if pd.notna(med_watts_climb) and pd.notna(weight_kg) and weight_kg > 0
        else np.nan
    )
    max_w_kg_climb = (
        float(max_watts_climb / weight_kg)
        if pd.notna(max_watts_climb) and pd.notna(weight_kg) and weight_kg > 0
        else np.nan
    )

    return {
        "n_rides_with_climb": int(valid_climb.count()),
        "total_climb_hours": float(rides_df["climb_time_min"].sum() / 60.0),
        "median_watts_climb": med_watts_climb,
        "max_watts_climb": max_watts_climb,
        "median_watts_climb_w_kg": med_w_kg_climb,
        "max_watts_climb_w_kg": max_w_kg_climb,
        "median_watts_flat": med_watts_flat,
        "median_climb_flat_power_ratio": med_ratio_cf,
    }


def extract_explosiveness_ratios(best_mmp: dict, cp_watts: float, median_avg_watts: float) -> dict:
    """Ratios d'explosivité sur 5s, 30s et 60s rapportés à la CP et à la puissance moyenne médiane."""
    out = {}
    for d in [5, 30, 60]:
        val = best_mmp.get(f"mmp_{d}s", np.nan)
        out[f"ratio_{d}s_cp"] = float(val / cp_watts) if (pd.notna(val) and pd.notna(cp_watts) and cp_watts > 0) else np.nan
        out[f"ratio_{d}s_avg_watts"] = (
            float(val / median_avg_watts)
            if (pd.notna(val) and pd.notna(median_avg_watts) and median_avg_watts > 0)
            else np.nan
        )
    return out


def extract_cardiac_drift_summary(rides_df: pd.DataFrame) -> dict:
    """Synthétise la dérive cardiaque (%) sur les sorties d'endurance (>= 45 min)."""
    if "cardiac_drift_pct" not in rides_df.columns:
        return {
            "n_rides_valid": 0,
            "median_drift_pct": np.nan,
            "iqr_drift_pct": np.nan,
            "pct_rides_low_drift_lt5": np.nan,
        }
    valid_drift = rides_df["cardiac_drift_pct"].dropna()
    if valid_drift.empty:
        return {
            "n_rides_valid": 0,
            "median_drift_pct": np.nan,
            "iqr_drift_pct": np.nan,
            "pct_rides_low_drift_lt5": np.nan,
        }
    q25, q75 = np.percentile(valid_drift, [25, 75])
    return {
        "n_rides_valid": int(valid_drift.count()),
        "median_drift_pct": float(valid_drift.median()),
        "iqr_drift_pct": float(q75 - q25),
        "pct_rides_low_drift_lt5": float((valid_drift < 5.0).mean() * 100.0),
    }


def compute_literature_normalized_scores(relative_metrics: dict) -> dict:
    """Calcule les ratios normalisés par rapport aux seuils P90 peloton pro (Valenzuela et al., 2022)."""
    scores = {}
    for key, ref_val in CAH_REFERENCE_BENCHMARKS.items():
        val = relative_metrics.get(key, np.nan)
        scores[f"{key}_pct_p90_valenzuela"] = float(val / ref_val) if pd.notna(val) else np.nan
    return scores


def summarize_window(rides_df: pd.DataFrame, weight_kg: float = np.nan, fc_max: float = np.nan) -> dict:
    """Synthétise le profil bioénergétique, Sig 1, Sig 2, explosivité, montée et dérive cardiaque."""
    if rides_df.empty:
        return {}

    best_mmp = {f"mmp_{d}s": float(rides_df[f"mmp_{d}s"].max()) for d in MMP_DURATIONS}
    cp, w_prime = fit_critical_power(best_mmp)
    median_avg_watts = float(rides_df["avg_watts"].dropna().median()) if "avg_watts" in rides_df.columns and not rides_df["avg_watts"].dropna().empty else np.nan

    relative_metrics = {}
    if pd.notna(weight_kg) and weight_kg > 0:
        for d in MMP_DURATIONS:
            val = best_mmp[f"mmp_{d}s"]
            relative_metrics[f"mmp_{d}s_w_kg"] = float(val / weight_kg) if pd.notna(val) else np.nan
        relative_metrics["cp_w_kg"] = float(cp / weight_kg) if pd.notna(cp) else np.nan
        relative_metrics["w_prime_j_kg"] = float(w_prime / weight_kg) if pd.notna(w_prime) else np.nan

    sig1_bioenergetic = classify_bioenergetic_valenzuela_dual(
        weight_kg=weight_kg,
        cp_watts=cp,
        w_prime_joules=w_prime,
    )
    sig2_4points = extract_sig2_4points(
        best_mmp=best_mmp,
        relative_metrics=relative_metrics,
        cp_watts=cp,
    )
    explosiveness = extract_explosiveness_ratios(best_mmp=best_mmp, cp_watts=cp, median_avg_watts=median_avg_watts)
    climbing_perf = extract_climbing_metrics(rides_df, weight_kg=weight_kg)
    drift_summary = extract_cardiac_drift_summary(rides_df)
    lit_scores = compute_literature_normalized_scores(relative_metrics)

    return {
        "n_rides": int(len(rides_df)),
        "date_start": rides_df["date"].min().strftime("%Y-%m-%d"),
        "date_end": rides_df["date"].max().strftime("%Y-%m-%d"),
        "total_hours": float(rides_df["moving_min"].sum() / 60.0),
        "total_distance_km": float(rides_df["distance_km"].sum()),
        "total_work_kj": float(rides_df["work_kj"].sum()),
        "median_avg_watts": median_avg_watts,
        "bioenergetics": {
            **best_mmp,
            **relative_metrics,
            "cp_watts": cp,
            "w_prime_joules": w_prime,
            "w_prime_kj": float(w_prime / 1000.0) if pd.notna(w_prime) else np.nan,
        },
        "signature_1_bioenergetic_valenzuela": sig1_bioenergetic,
        "signature_2_4points_5s_30s_60s_cp": sig2_4points,
        "explosiveness_ratios": explosiveness,
        "climbing_performance": climbing_perf,
        "cardiac_drift": drift_summary,
        "literature_normalized_scores": lit_scores,
    }

## 4. Construction des profils sur les 7 cyclistes (`construire_profils`)

In [ ]:
def construire_profils(dossier_sorties: Path = DOSSIER_SORTIES) -> tuple[dict, pd.DataFrame, dict]:
    all_athletes_data = {}
    all_rides_by_athlete = {}
    rows_for_cah = []

    for dossier in sorted(d for d in dossier_sorties.iterdir() if d.is_dir()):
        m = re.match(r"(cyclist\d+)(?:_(\d+)kg)?", dossier.name)
        cycliste = m.group(1) if m else dossier.name
        poids_cycliste = int(m.group(2)) if m and m.group(2) else None
        fc_repos = lire_fc_repos(dossier)

        fichiers = sorted(f for f in dossier.rglob("*.csv") if "fc_repos" not in f.name)

        def _safe_read(f: Path) -> dict:
            try:
                return lire_sortie_profil(f)
            except Exception as e:
                print(f"[{cycliste}] erreur {f.name} : {e}")
                return {}

        with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
            results = list(executor.map(_safe_read, fichiers))

        sorties = [r for r in results if r]
        if not sorties:
            continue

        rides_df = pd.DataFrame(sorties).sort_values(["date", "seance"]).reset_index(drop=True)
        all_rides_by_athlete[cycliste] = rides_df

        fc_max = FC_MAX.get(cycliste) or float(rides_df["hr_max"].max())
        weight_val = float(poids_cycliste) if poids_cycliste is not None else np.nan

        global_summary = summarize_window(rides_df, weight_kg=weight_val, fc_max=fc_max)

        max_date = rides_df["date"].max()
        recent_df = rides_df[rides_df["date"] >= (max_date - pd.Timedelta(days=RECENT_WINDOW_DAYS))]
        recent_summary = summarize_window(recent_df, weight_kg=weight_val, fc_max=fc_max)

        by_year = {
            str(int(yr)): summarize_window(grp, weight_kg=weight_val, fc_max=fc_max)
            for yr, grp in rides_df.groupby("annee")
        }

        profil_choisi = PROFILS_CHOISIS.get(cycliste, "non_defini")

        all_athletes_data[cycliste] = {
            "athlete_id": cycliste,
            "profil_choisi": profil_choisi,
            "info": {
                "poids_kg": poids_cycliste,
                "fc_max": fc_max,
                "fc_repos": fc_repos,
                "annees": sorted({int(a) for a in rides_df["annee"].dropna()}),
                "nb_fichiers_lus": len(fichiers),
                "nb_seances_valides": len(rides_df),
            },
            "global_profile": global_summary,
            "recent_90d_profile": recent_summary,
            "annual_profiles": by_year,
        }

        bio = global_summary["bioenergetics"]
        sig1_raw = global_summary["signature_1_bioenergetic_valenzuela"]["raw"]
        sig1_rel = global_summary["signature_1_bioenergetic_valenzuela"]["relative"]
        exp_r = global_summary["explosiveness_ratios"]
        climb = global_summary["climbing_performance"]
        drift = global_summary["cardiac_drift"]

        # Ensemble des indicateurs physiologiques clés pour la CAH multi-dimensionnelle
        rows_for_cah.append({
            "athlete_id": cycliste,
            "profil_choisi": profil_choisi,
            "poids_kg": weight_val,
            "cp_watts": bio.get("cp_watts", np.nan),
            "cp_w_kg": bio.get("cp_w_kg", np.nan),
            "w_prime_kj": bio.get("w_prime_kj", np.nan),
            "w_prime_j_kg": bio.get("w_prime_j_kg", np.nan),
            "mmp_5s_w": bio.get("mmp_5s", np.nan),
            "mmp_30s_w": bio.get("mmp_30s", np.nan),
            "mmp_60s_w": bio.get("mmp_60s", np.nan),
            "mmp_5s_w_kg": bio.get("mmp_5s_w_kg", np.nan),
            "mmp_30s_w_kg": bio.get("mmp_30s_w_kg", np.nan),
            "mmp_60s_w_kg": bio.get("mmp_60s_w_kg", np.nan),
            "ratio_5s_cp": exp_r.get("ratio_5s_cp", np.nan),
            "ratio_30s_cp": exp_r.get("ratio_30s_cp", np.nan),
            "ratio_60s_cp": exp_r.get("ratio_60s_cp", np.nan),
            "watts_climb_med": climb.get("median_watts_climb", np.nan),
            "w_kg_climb_med": climb.get("median_watts_climb_w_kg", np.nan),
            "climb_flat_ratio": climb.get("median_climb_flat_power_ratio", np.nan),
            "cardiac_drift_med_pct": drift.get("median_drift_pct", np.nan),
            "sig1_raw_dominant": sig1_raw.get("dominant_profile"),
            "sig1_raw_prob": sig1_raw.get("dominant_probability_pct", np.nan),
            "sig1_rel_dominant": sig1_rel.get("dominant_profile"),
            "sig1_rel_prob": sig1_rel.get("dominant_probability_pct", np.nan),
        })

    cah_df = pd.DataFrame(rows_for_cah)
    return all_athletes_data, cah_df, all_rides_by_athlete


all_athletes_data, cah_df, all_rides_by_athlete = construire_profils()

# Tableau récapitulatif épuré
summary_cols = [
    "athlete_id",
    "profil_choisi",
    "poids_kg",
    "cp_watts",
    "cp_w_kg",
    "w_prime_kj",
    "w_prime_j_kg",
    "w_kg_climb_med",
    "climb_flat_ratio",
    "cardiac_drift_med_pct",
    "sig1_raw_dominant",
    "sig1_rel_dominant",
]
display(cah_df[summary_cols].round(3))

## 4b. Signature 1 — Comparaison horizontale de la classification Brute vs Relative (%)

In [ ]:
archetype_order = list(VALENZUELA_5_ARCHETYPES.keys())
archetype_colors = {
    "Grimpeur": "#2b6cb0",
    "Rouleur_CLM": "#2f855a",
    "Polyvalent_Puncheur": "#dd6b20",
    "Sprinteur": "#c53030",
    "Classement_General": "#805ad5",
}

athletes_sorted = sorted(all_athletes_data.keys())
y_labels = [
    f"{cid} ({all_athletes_data[cid].get('profil_choisi', '')})"
    for cid in athletes_sorted
]
y_pos = np.arange(len(athletes_sorted))

fig, (ax_raw_bar, ax_rel_bar) = plt.subplots(1, 2, figsize=(15.5, 5.8), sharey=True)

for ax, mode_key, title_txt in [
    (ax_raw_bar, "raw", "Signature 1 — Classification Brute (Poids, CP [W], W' [kJ])"),
    (ax_rel_bar, "relative", "Signature 1 — Classification Relative (Poids, CP [W/kg], W' [J/kg])"),
]:
    left_cum = np.zeros(len(athletes_sorted))
    for arch in archetype_order:
        vals = np.array([
            all_athletes_data[cid]["global_profile"]["signature_1_bioenergetic_valenzuela"][mode_key]
            .get("membership_probabilities_pct", {})
            .get(arch, 0.0)
            for cid in athletes_sorted
        ], dtype=float)
        vals = np.nan_to_num(vals, nan=0.0)

        bars = ax.barh(
            y_pos,
            vals,
            left=left_cum,
            color=archetype_colors[arch],
            edgecolor="white",
            height=0.62,
            label=arch.replace("_", " "),
        )
        for idx_b, w_val in enumerate(vals):
            if w_val >= 8.0:
                ax.text(
                    left_cum[idx_b] + w_val / 2.0,
                    y_pos[idx_b],
                    f"{w_val:.1f}%",
                    ha="center",
                    va="center",
                    color="white",
                    fontsize=8.5,
                    fontweight="bold",
                )
        left_cum += vals

    ax.set_xlim(0, 100)
    ax.set_xlabel("Probabilité d'appartenance Softmax (%)")
    ax.set_title(title_txt, fontsize=11.5, fontweight="bold")
    ax.grid(True, axis="x", linestyle="--", alpha=0.35)

ax_raw_bar.set_yticks(y_pos)
ax_raw_bar.set_yticklabels(y_labels, fontsize=9.5)
ax_raw_bar.invert_yaxis()

handles, labels = ax_rel_bar.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=5, frameon=True, fontsize=9)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(OUTPUT_DIR / "signature1_comparaison_brut_vs_relatif_pct.png", dpi=160, bbox_inches="tight")
plt.show()

## 5. Signature 2 — Profils de puissance courte durée (`5 s`, `30 s`, `60 s`, `CP`) : Brut (W) et Relatif (W/kg)

In [ ]:
points = ["5s", "30s", "60s", "CP"]
x_idx = np.arange(len(points))

fig, (ax_raw, ax_rel) = plt.subplots(1, 2, figsize=(14.5, 5.6))
cmap = plt.get_cmap("tab10")

for idx, (cycliste, payload) in enumerate(sorted(all_athletes_data.items())):
    sig2 = payload["global_profile"]["signature_2_4points_5s_30s_60s_cp"]
    y_raw = [sig2["raw_watts"].get(p, np.nan) for p in points]
    y_rel = [sig2["relative_w_kg"].get(p, np.nan) for p in points]
    p_choisi = payload.get("profil_choisi", "")
    lbl = f"{cycliste} ({p_choisi})" if p_choisi else cycliste
    color = cmap(idx % 10)

    ax_raw.plot(x_idx, y_raw, marker="o", linewidth=2.2, markersize=6.5, color=color, label=lbl)
    ax_rel.plot(x_idx, y_rel, marker="s", linewidth=2.2, markersize=6.5, color=color, label=lbl)

ax_raw.set_title("Signature 2 — Puissance courte durée & CP (Brut, W)", fontsize=12, fontweight="bold")
ax_raw.set_xticks(x_idx)
ax_raw.set_xticklabels(["5 s", "30 s", "60 s", "CP"])
ax_raw.set_ylabel("Puissance (W)")
ax_raw.grid(True, linestyle="--", alpha=0.35)
ax_raw.legend(loc="upper right", fontsize=8.5, frameon=True)

ax_rel.set_title("Signature 2 — Puissance courte durée & CP (Relatif, W/kg)", fontsize=12, fontweight="bold")
ax_rel.set_xticks(x_idx)
ax_rel.set_xticklabels(["5 s", "30 s", "60 s", "CP"])
ax_rel.set_ylabel("Puissance relative (W/kg)")
ax_rel.grid(True, linestyle="--", alpha=0.35)
ax_rel.legend(loc="upper right", fontsize=8.5, frameon=True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "signature2_4points_brut_relatif.png", dpi=160)
plt.show()

## 6. Signature 3 — Performance en montée (`slope > 4 %`) vs plat (`-1 % <= slope <= 1 %`)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.2))
plot_df = cah_df.sort_values("w_kg_climb_med", ascending=False).reset_index(drop=True)
x_pos = np.arange(len(plot_df))
labels_x = [f"{r['athlete_id']}\n({r['profil_choisi']})" for _, r in plot_df.iterrows()]

bars1 = ax1.bar(x_pos, plot_df["w_kg_climb_med"], color="#2b6cb0", alpha=0.88)
ax1.set_title("Puissance médiane en montée (slope > 4 %) — W/kg", fontsize=12, fontweight="bold")
ax1.set_xticks(x_pos)
ax1.set_xticklabels(labels_x, fontsize=8.5)
ax1.set_ylabel("Puissance en montée (W/kg)")
ax1.grid(True, axis="y", linestyle="--", alpha=0.35)
for b in bars1:
    h = b.get_height()
    if pd.notna(h):
        ax1.text(b.get_x() + b.get_width() / 2, h + 0.05, f"{h:.2f}", ha="center", va="bottom", fontsize=8.5)

bars2 = ax2.bar(x_pos, plot_df["climb_flat_ratio"], color="#2f855a", alpha=0.88)
ax2.axhline(1.0, color="#718096", linestyle=":", linewidth=1.2, label="Ratio = 1.0 (même puissance montée/plat)")
ax2.set_title("Ratio puissance Montée (>4 %) / Plat (-1..1 %)", fontsize=12, fontweight="bold")
ax2.set_xticks(x_pos)
ax2.set_xticklabels(labels_x, fontsize=8.5)
ax2.set_ylabel("Ratio Puissance Montée / Plat")
ax2.grid(True, axis="y", linestyle="--", alpha=0.35)
ax2.legend(fontsize=8.5)
for b in bars2:
    h = b.get_height()
    if pd.notna(h):
        ax2.text(b.get_x() + b.get_width() / 2, h + 0.02, f"{h:.2f}", ha="center", va="bottom", fontsize=8.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "signature3_montee_vs_plat.png", dpi=160)
plt.show()

## 7. Évolution de la dérive cardiaque (`Pw:Hr`) par sortie et comparaison entre athlètes

In [ ]:
fig, (ax_traj, ax_box) = plt.subplots(1, 2, figsize=(15, 5.6), gridspec_kw={"width_ratios": [1.45, 1.0]})
cmap = plt.get_cmap("tab10")

drift_series_list = []
drift_box_labels = []

sorted_athletes = sorted(all_athletes_data.keys())
for idx, cycliste in enumerate(sorted_athletes):
    rides = all_rides_by_athlete.get(cycliste, pd.DataFrame())
    if rides.empty or "cardiac_drift_pct" not in rides.columns:
        continue
    d_df = rides.dropna(subset=["cardiac_drift_pct"]).sort_values("date").reset_index(drop=True)
    if d_df.empty:
        continue

    p_choisi = all_athletes_data[cycliste].get("profil_choisi", "")
    lbl = f"{cycliste} ({p_choisi})" if p_choisi else cycliste
    color = cmap(idx % 10)

    x_rides = np.arange(1, len(d_df) + 1)
    y_drift = d_df["cardiac_drift_pct"].to_numpy(dtype=float)
    ax_traj.plot(
        x_rides,
        y_drift,
        marker="o",
        markersize=4.5,
        linewidth=1.6,
        alpha=0.85,
        color=color,
        label=f"{lbl} (méd. {np.median(y_drift):+.1f}%)",
    )

    drift_series_list.append(y_drift)
    drift_box_labels.append(f"{cycliste}\n({p_choisi})")

ax_traj.axhline(5.0, color="#d69e2e", linestyle="--", linewidth=1.2, label="Seuil 5 % (endurance stable)")
ax_traj.axhline(0.0, color="#718096", linestyle=":", linewidth=1.0)
ax_traj.set_title("Dérive cardiaque par sortie d'endurance (>= 45 min)", fontsize=12, fontweight="bold")
ax_traj.set_xlabel("Numéro de sortie d'endurance valide (ordre chronologique)")
ax_traj.set_ylabel("Dérive cardiaque Pw:Hr (%)")
ax_traj.grid(True, linestyle="--", alpha=0.35)
ax_traj.legend(loc="best", fontsize=8, frameon=True)

bp = ax_box.boxplot(drift_series_list, tick_labels=drift_box_labels, patch_artist=True, showfliers=True)
for idx, patch in enumerate(bp["boxes"]):
    patch.set_facecolor(cmap(idx % 10))
    patch.set_alpha(0.65)
for median_line in bp["medians"]:
    median_line.set_color("black")
    median_line.set_linewidth(1.8)

ax_box.axhline(5.0, color="#d69e2e", linestyle="--", linewidth=1.2)
ax_box.axhline(0.0, color="#718096", linestyle=":", linewidth=1.0)
ax_box.set_title("Distribution comparée de la dérive cardiaque (%)", fontsize=12, fontweight="bold")
ax_box.set_ylabel("Dérive cardiaque Pw:Hr (%)")
ax_box.tick_params(axis="x", labelsize=8.5, rotation=20)
ax_box.grid(True, axis="y", linestyle="--", alpha=0.35)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "evolution_derive_cardiaque.png", dpi=160)
plt.show()

## 8. Classification Ascendante Hiérarchique (CAH – Ward) sur l'ensemble des indicateurs de profilage

In [ ]:
cah_features = [
    "poids_kg",
    "cp_w_kg",
    "w_prime_j_kg",
    "mmp_5s_w_kg",
    "mmp_30s_w_kg",
    "mmp_60s_w_kg",
    "ratio_5s_cp",
    "ratio_30s_cp",
    "ratio_60s_cp",
    "w_kg_climb_med",
    "climb_flat_ratio",
    "cardiac_drift_med_pct",
]

X_raw = cah_df[cah_features].copy()
for col in cah_features:
    X_raw[col] = X_raw[col].fillna(X_raw[col].median())

means = X_raw.mean(axis=0)
stds = X_raw.std(axis=0, ddof=0).replace(0, 1.0)
X_std = ((X_raw - means) / stds).to_numpy(dtype=float)

# Linkage de Ward sur distances euclidiennes standardisées
Z_ward = linkage(X_std, method="ward")

# Étiquettes du dendrogramme : cyclistX (profil choisi)
dendro_labels = [
    f"{row['athlete_id']} ({row['profil_choisi']})"
    for _, row in cah_df.iterrows()
]

fig, (ax_dend, ax_mat) = plt.subplots(1, 2, figsize=(15.5, 5.8), gridspec_kw={"width_ratios": [1.2, 1.0]})

dendrogram(
    Z_ward,
    labels=dendro_labels,
    orientation="right",
    leaf_font_size=10,
    color_threshold=0.7 * float(Z_ward[-1, 2]),
    ax=ax_dend,
)
ax_dend.set_title("Dendrogramme CAH (Ward) — Indicateurs complets de profilage", fontsize=12, fontweight="bold")
ax_dend.set_xlabel("Distance euclidienne agrégée (Ward)")
ax_dend.grid(True, axis="x", linestyle="--", alpha=0.35)

dist_mat = squareform(pdist(X_std, metric="euclidean"))
leaf_order = dendrogram(Z_ward, labels=dendro_labels, no_plot=True)["leaves"]
dist_ordered = dist_mat[np.ix_(leaf_order, leaf_order)]
labels_ordered = [dendro_labels[i] for i in leaf_order]

im = ax_mat.imshow(dist_ordered, cmap="YlGnBu", aspect="auto")
ax_mat.set_xticks(np.arange(len(labels_ordered)))
ax_mat.set_yticks(np.arange(len(labels_ordered)))
ax_mat.set_xticklabels(labels_ordered, rotation=35, ha="right", fontsize=8.5)
ax_mat.set_yticklabels(labels_ordered, fontsize=8.5)
ax_mat.set_title("Distances euclidiennes standardisées (Z-score)", fontsize=12, fontweight="bold")
for i in range(len(labels_ordered)):
    for j in range(len(labels_ordered)):
        val = dist_ordered[i, j]
        txt_color = "white" if val > (dist_ordered.max() * 0.55) else "black"
        ax_mat.text(j, i, f"{val:.1f}", ha="center", va="center", color=txt_color, fontsize=8)
fig.colorbar(im, ax=ax_mat, fraction=0.046, pad=0.04, label="Distance Z")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cah_ward_dendrogramme_profils.png", dpi=160)
plt.show()

## 9. Export JSON strict (Cohorte globale, CAH Ward & Profils individuels)

In [ ]:
def _sanitize_for_json(obj):
    if isinstance(obj, dict):
        return {str(k): _sanitize_for_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_sanitize_for_json(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        val = float(obj)
        return None if (math.isnan(val) or math.isinf(val)) else val
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if obj is None or (isinstance(obj, float) and pd.isna(obj)):
        return None
    return obj


def export_profiles_json(
    athletes_data: dict,
    cah_dataframe: pd.DataFrame,
    linkage_matrix: np.ndarray,
    cah_feat_cols: list[str],
    dendrogram_lbls: list[str],
    output_dir: Path = OUTPUT_DIR,
) -> dict:
    output_dir.mkdir(parents=True, exist_ok=True)

    cah_records = (
        cah_dataframe.to_dict(orient="records")
        if cah_dataframe is not None and not cah_dataframe.empty
        else []
    )

    cohort_payload = _sanitize_for_json({
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "n_athletes": len(athletes_data),
            "literature_reference": "Valenzuela et al. (2022) — 5 archetypes & P90 pro benchmarks",
            "climbing_slope_threshold_pct": SLOPE_CLIMB_MIN,
            "flat_slope_range_pct": [-SLOPE_FLAT_ABS, SLOPE_FLAT_ABS],
            "cardiac_drift_min_duration_s": MIN_DRIFT_DURATION_S,
        },
        "valenzuela_5_archetypes_raw": VALENZUELA_5_ARCHETYPES,
        "valenzuela_5_archetypes_relative_delta": VALENZUELA_5_ARCHETYPES_REL,
        "cah_reference_benchmarks_p90": CAH_REFERENCE_BENCHMARKS,
        "profils_choisis": PROFILS_CHOISIS,
        "cah_summary_table": cah_records,
        "cah_ward": {
            "features_used": cah_feat_cols,
            "athlete_order": cah_dataframe["athlete_id"].tolist() if cah_dataframe is not None else [],
            "dendrogram_labels": dendrogram_lbls,
            "linkage_matrix_ward": linkage_matrix.tolist() if linkage_matrix is not None else [],
        },
        "athletes": athletes_data,
    })

    cohort_path = output_dir / "cohort_profiles.json"
    with open(cohort_path, "w", encoding="utf-8") as f:
        json.dump(cohort_payload, f, indent=2, ensure_ascii=False, allow_nan=False)

    exported_files = {"cohort": str(cohort_path)}

    for athlete_id, payload in athletes_data.items():
        athlete_payload = _sanitize_for_json({
            "metadata": {
                "generated_at": datetime.now().isoformat(),
                "athlete_id": athlete_id,
            },
            **payload,
        })
        athlete_path = output_dir / f"{athlete_id}_profile.json"
        with open(athlete_path, "w", encoding="utf-8") as f:
            json.dump(athlete_payload, f, indent=2, ensure_ascii=False, allow_nan=False)
        exported_files[athlete_id] = str(athlete_path)

    return exported_files


exported_paths = export_profiles_json(
    athletes_data=all_athletes_data,
    cah_dataframe=cah_df,
    linkage_matrix=Z_ward,
    cah_feat_cols=cah_features,
    dendrogram_lbls=dendro_labels,
    output_dir=OUTPUT_DIR,
)
print("Fichiers JSON exportés :", json.dumps(exported_paths, indent=2, ensure_ascii=False))